1. Loaded the document.
2. Cleaned the text.
3. Split it into chunks.
4. Converted chunks into embeddings.
5. Stored them in FAISS.
6. Retrival.

### ---------------------------------------------------------------------------------

- `from langchain_text_splitters import RecursiveCharacterTextSplitter` 

    - LLMs cannot process huge documents at once because they have a context limit.

    - So we divide a large document into smaller pieces called chunks.

    (Recursive chunking)

- `from langchain_community.vectorstores import FAISS`
 
    - it stores vector

- `from langchain_huggingface import HuggingFaceEmbeddings`

    - convert text into vectors

- `from langchain_google_genai import ChatGoogleGenerativeAI`

    - Used to connect with Google's Gemini model

- `from nltk.tokenize import word_tokenize`

    - Splits a sentence into words

- `import spacy`

    - SpaCy is used for NLP.

    - it is used for Tokenization , Lemmatization , Stopword Removal

- `import re`

    - Used for text cleaning

- `import contractions`

    - Expands contractions

- `from textblob import TextBlob`

    - Used for spelling correction

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from nltk.tokenize import word_tokenize
from textblob import TextBlob
import contractions
import spacy
import re   

### 1. Load the document (.txt)

In [4]:
data=open('data.txt').read()

### 2. Text Normalization


#### a. converting all the character into lowercase

In [5]:
data=data.lower()

#### b. removing Extra space

In [6]:
data=re.sub(r'\s{2,}',' ',data)

- removing numbers like 1.

In [7]:
data=re.sub(r'\d+\.','',data)
data

"machine learning is a branch of artificial intelligence that focuses on enabling computers to learn patterns from data.\ninstead of explicitly programming every rule, machine learning algorithms learn from examples and use those examples to make predictions.\nmachine learning has become an important technology in many industries.\nit is used in healthcare, finance, education, transportation, retail, entertainment, and cybersecurity.\nthe basic idea behind machine learning is simple.\nwe provide data to an algorithm, allow the algorithm to learn patterns, and then use the trained model to make predictions on new data.\nmachine learning can be divided into several major categories.\nthe most common categories are supervised learning, unsupervised learning, semi-supervised learning, and reinforcement learning.\neach type of learning solves a different kind of problem.\nthe choice of learning method depends on the type of data available and the objective of the project.\nsupervised learni

#### c. Contraction

In [8]:
data=contractions.fix(data)  #check the return type before use

#### d. Removing the punctuation and spl characters

In [9]:
data=re.sub(r'[^0-9a-zA-Z\s]','',data)
data

'machine learning is a branch of artificial intelligence that focuses on enabling computers to learn patterns from data\ninstead of explicitly programming every rule machine learning algorithms learn from examples and use those examples to make predictions\nmachine learning has become an important technology in many industries\nit is used in healthcare finance education transportation retail entertainment and cybersecurity\nthe basic idea behind machine learning is simple\nwe provide data to an algorithm allow the algorithm to learn patterns and then use the trained model to make predictions on new data\nmachine learning can be divided into several major categories\nthe most common categories are supervised learning unsupervised learning semisupervised learning and reinforcement learning\neach type of learning solves a different kind of problem\nthe choice of learning method depends on the type of data available and the objective of the project\nsupervised learning is a type of machine

#### e. Textblob

- we use textblob to correct the sentence ie. spelling correction

In [10]:
# data = str(TextBlob(data).correct())
# values=TextBlob(data).correct().raw_sentences
# data=' '.join(values)


#### f. Spacy lamentization

What is SpaCy?

- SpaCy is an NLP (Natural Language Processing) library.

- It helps computers understand English language

- It can perform:

        Tokenization
        Lemmatization
        Stop word removal
        POS tagging
        Named Entity Recognition
        Dependency Parsing

- in this project used for stop word removal

In [ ]:
nlp=spacy.load('en_core_web_sm')   #It is SpaCy's small English language model (for grammer training)
tokens=nlp(data)   #This line sends the sentence into SpaCy.

print(type(tokens))  #op tional
'''A Doc object is a container that stores the processed sentence '''

update_tokens=[token.lemma_ for token in tokens if not token.is_stop]  #token.lemma_ → human-readable text
data=' '.join(update_tokens).strip()  # we use strip to extra spaces

<class 'spacy.tokens.doc.Doc'>


#### g. chunking(converting the doc into chunks [doc-->chunk])

- Its job is to split a long document into smaller chunks.

In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)

# Generate chunks from the lemmatized text
'''chunks = splitter.split_text(data)'''  # it return all chunks
chunks = splitter.create_documents([data])
chunks

<class 'list'>


[Document(metadata={}, page_content='machine learning branch artificial intelligence focus enable computer learn pattern datum \n instead explicitly program rule machine learning algorithm learn example use example prediction'),
 Document(metadata={}, page_content='machine learning important technology industry \n healthcare finance education transportation retail entertainment cybersecurity \n basic idea machine learning simple'),
 Document(metadata={}, page_content='basic idea machine learning simple \n provide datum algorithm allow algorithm learn pattern use train model prediction new datum \n machine learning divide major category'),
 Document(metadata={}, page_content='common category supervise learn unsupervised learning semisupervise learning reinforcement learn \n type learning solve different kind problem'),
 Document(metadata={}, page_content='choice learn method depend type datum available objective project \n supervise learning type machine learning model learn label datum

In [13]:
print(chunks[0].page_content)
# print(type(chunks[0]))

machine learning branch artificial intelligence focus enable computer learn pattern datum 
 instead explicitly program rule machine learning algorithm learn example use example prediction


In [14]:
chunks[0].metadata={'file_name':'data.txt'}  # we add informations using the meta data
chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='machine learning branch artificial intelligence focus enable computer learn pattern datum \n instead explicitly program rule machine learning algorithm learn example use example prediction'),
 Document(metadata={}, page_content='machine learning important technology industry \n healthcare finance education transportation retail entertainment cybersecurity \n basic idea machine learning simple'),
 Document(metadata={}, page_content='basic idea machine learning simple \n provide datum algorithm allow algorithm learn pattern use train model prediction new datum \n machine learning divide major category'),
 Document(metadata={}, page_content='common category supervise learn unsupervised learning semisupervise learning reinforcement learn \n type learning solve different kind problem'),
 Document(metadata={}, page_content='choice learn method depend type datum available objective project \n supervise learning type machine learning 

In [ ]:
print(len(chunks))
#chunks --> list of documents

141


#### h. chunk embeddings (converting the chunks to vectors)

In [ ]:
embedding_model=HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-miniLM-L6-v2'  #to convert the chunks into vector
) 
# Every sentence becomes a list of numbers

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

vector db

- FAISS.from_documents  it does 3 things

        Read every document

        convert chunk into vectors

        Store vectors inside FAISS

In [17]:
vectordb=FAISS.from_documents(documents=chunks,embedding=embedding_model)
vectordb

### Retrival 

In [18]:
user_query='what is machine learning?'
r_chunks=vectordb.similarity_search(user_query)  #directly we will get doc



### combining the sentence

In [19]:
updated_r_chunks=set()

for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)

r_text='\n'.join(updated_r_chunks)
r_text


'machine learning branch artificial intelligence focus enable computer learn pattern datum \n instead explicitly program rule machine learning algorithm learn example use example prediction\nskill contribute successful machine learning project machine learning continue evolve rapidly \n new algorithm architecture introduce regularly \n fundamental principle remain \n good datum essential\nmachine learning important technology industry \n healthcare finance education transportation retail entertainment cybersecurity \n basic idea machine learning simple\nbasic idea machine learning simple \n provide datum algorithm allow algorithm learn pattern use train model prediction new datum \n machine learning divide major category'

#### used hugging face

In [ ]:
#retrival
def r_search(query,k=3):    
    R_chunks =vectordb.similarity_search(query)
    # print(R_chunks)
    R_chunks={doc.page_content for doc in R_chunks}  # we have used the set it will remove the duplicate
    R_Text='\n'.join(R_chunks)
    # print(R_Text)
    return R_Text
# generation
def g_text(r_search,query):
        import os
        prompt = f'''
                    You're an helpful assistant
                    Assigned Task for you : Structure my output => {r_search}
                    for the input=> {query}
                    Note : 
                    1) Don't add extra contents just structure mentioned output.
                    2) If there is mistake in output correct or else keep the original output
                    with structured result
                    Output structure:
                    Input :{query}
                    output :structured output
            '''
        llm_model=ChatGoogleGenerativeAI(
            model='gemini-3.5-flash', 
            api_key=os.environ['GEMINI_API_KEY'] #gemini-3.6-flash , gemini-2.0-flash, gemini-2.5-flash
            temperature=0.0 #important
            )
        response=llm_model.invoke(prompt).content
        return response
user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt) # text normalization

r_response = r_search(user_prompt)
g_response = g_text(r_response,user_prompt)
print(g_response)

[{'type': 'text', 'text': 'Input : Explain Machine Learning \noutput :\n\n**Definition**\n* Machine learning is a branch of artificial intelligence focused on enabling computers to learn patterns from data. \n* Instead of explicitly programming rules, machine learning algorithms learn from examples and use those examples to make predictions.\n\n**The Basic Idea**\n* The basic idea of machine learning is simple: provide data to an algorithm, allow the algorithm to learn patterns, and use this trained model to make predictions on new data.\n\n**Evolution and Fundamental Principles**\n* Skills contribute to successful machine learning projects. \n* Machine learning continues to evolve rapidly, and new algorithms and architectures are introduced regularly. \n* However, the fundamental principles remain: having good data is essential.\n\n**Industry Applications**\n* Machine learning is an important technology across various industries, including:\n  * Healthcare\n  * Finance\n  * Education\

In [ ]:
def r_search(query,k=3):      # Return the top 3 most similar chunks.
    R_chunks =vectordb.similarity_search(query)
    R_chunks={doc.page_content for doc in R_chunks}  # we have used the set it will remove the duplicate
    R_Text='\n'.join(R_chunks)
    return R_Text


# generation
def g_text(r_search,query):
        import os
        prompt = f'''
                    You're an helpful assistant
                    Assigned Task for you : Structure my output => {r_search}
                    for the input=> {query}
                    Note : 
                    1) Don't add extra contents just structure mentioned output.
                    2) If there is mistake in output correct or else keep the original output
                    with structured result
                    Output structure:
                    Input :{query}
                    output :structured output 
            '''
        llm_model=ChatGoogleGenerativeAI(
            model='gemini-3.5-flash', api_key=os.environ['GEMINI_API_KEY'] #gemini-3.6-flash , gemini-2.0-flash, gemini-2.5-flash
            )
        response=llm_model.invoke(prompt).content
        return response

user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)   # text normalization

r_response = r_search(user_prompt)
g_response = g_text(r_response,user_prompt)
print(g_response)

In [23]:
print(r_chunks)

[Document(id='cac2e708-2a92-4e64-a887-ce65fbae20a5', metadata={}, page_content='machine learning important technology industry \n healthcare finance education transportation retail entertainment cybersecurity \n basic idea machine learning simple'), Document(id='4ea636ed-d68c-48d5-9288-d9edbbfb40bc', metadata={}, page_content='skill contribute successful machine learning project machine learning continue evolve rapidly \n new algorithm architecture introduce regularly \n fundamental principle remain \n good datum essential'), Document(id='6307c05b-09f5-46af-a43f-0286a8b93059', metadata={'file_name': 'data.txt'}, page_content='machine learning branch artificial intelligence focus enable computer learn pattern datum \n instead explicitly program rule machine learning algorithm learn example use example prediction'), Document(id='a47d258b-3df2-4e01-bcca-320aebca8319', metadata={}, page_content='basic idea machine learning simple \n provide datum algorithm allow algorithm learn pattern use

In [21]:
print(g_response[0]['text'])

Input : Explain Machine Learning 
output :

**Definition**
* Machine learning is a branch of artificial intelligence focused on enabling computers to learn patterns from data. 
* Instead of explicitly programming rules, machine learning algorithms learn from examples and use those examples to make predictions.

**The Basic Idea**
* The basic idea of machine learning is simple: provide data to an algorithm, allow the algorithm to learn patterns, and use this trained model to make predictions on new data.

**Evolution and Fundamental Principles**
* Skills contribute to successful machine learning projects. 
* Machine learning continues to evolve rapidly, and new algorithms and architectures are introduced regularly. 
* However, the fundamental principles remain: having good data is essential.

**Industry Applications**
* Machine learning is an important technology across various industries, including:
  * Healthcare
  * Finance
  * Education
  * Transportation
  * Retail
  * Entertainmen

Input :Explain Machine Learning 

output :

**What is Machine Learning?**
* Machine learning is a branch of artificial intelligence focused on enabling computers to learn patterns from data.
* Instead of explicitly programming rules, machine learning algorithms learn from examples and use these examples to make predictions.
* The basic idea of machine learning is simple.

**How Machine Learning Works**
* You provide data to an algorithm, allowing the algorithm to learn patterns.
* You then use the trained model to make predictions on new data.

**Categories of Machine Learning**
* Machine learning is divided into major categories.

**The Machine Learning Lifecycle**
The machine learning lifecycle includes several stages:
1. The process begins with understanding the business problem.
2. The next step is to collect relevant data.

**Explainable AI**
* Explainable AI techniques attempt to provide insights into model predictions.

**Importance in Industry**
...
* Transportation
* Retail
* Entertainment
* Cybersecurity